# Prepare 2024 Train / Test Splits

- Đọc 4 file parquet: `24_01`, `24_02`, `24_03`, `24_04`
- Giữ lại các cột cần thiết
- **Train**: tháng 1–3 | **Test**: tháng 4
- PCA trên embedding (95% variance) — fit trên train, transform train & test
- StandardScaler — fit trên train, transform train & test
- Lưu `train.parquet` và `test.parquet` vào thư mục `data/`

In [31]:
import os
import numpy as np
import pandas as pd
import joblib
from dotenv import dotenv_values
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [32]:

# ── Config ─────────────────────────────────────────────────────────────────
ENV_PATH       = r"C:\Users\Admin\Documents\Tuan_Huy\MOE\.env"
OUTPUT_DIR     = r"C:\Users\Admin\Documents\Tuan_Huy\MOE\data"

EMBEDDING_COL  = "embedding"
TARGET_COLS    = ["duration", "avgpcon", "pclass", "ec"]   # ec is derived from EXIT_STATE_COL
EXIT_STATE_COL = "exit state"   # raw column in source data → transformed to binary `ec`
ADT_COL        = "adt"
QDT_COL        = "qdt"
NUMERIC_FEATS  = ["cnumr", "nnumr", "pri", "freq_req", "mszl", "elpl"]

PCA_VARIANCE   = 0.80   # giữ lại 95% variance
RANDOM_SEED    = 42

# Note: "ec" is derived, not read directly; EXIT_STATE_COL is read and transformed
RAW_TARGET_COLS = ["duration", "avgpcon", "pclass", EXIT_STATE_COL]
KEEP_COLS = [EMBEDDING_COL] + RAW_TARGET_COLS + [ADT_COL, QDT_COL] + NUMERIC_FEATS

config    = dotenv_values(ENV_PATH)
DATA_PATH = config["PATH"]
print(f"DATA_PATH : {DATA_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)


DATA_PATH : C:\Users\Admin\Documents\Tuan_Huy\DREAM-2026-workshop-SCA-HPCAsia\F-DATA\data
OUTPUT_DIR: C:\Users\Admin\Documents\Tuan_Huy\MOE\data


## 1. Đọc & gắn nhãn tháng

In [33]:
months = [1, 2, 3, 4]
frames = []

for m in months:
    fpath = os.path.join(DATA_PATH, f"24_{m:02d}.parquet")
    print(f"Reading: {fpath}")
    df_m = pd.read_parquet(fpath)

    # Chỉ giữ các cột cần thiết (bỏ qua cột không tồn tại)
    cols_present = [c for c in KEEP_COLS if c in df_m.columns]
    missing = set(KEEP_COLS) - set(cols_present)
    if missing:
        print(f"  [WARN] Tháng {m}: thiếu cột {missing}")
    df_m = df_m[cols_present].copy()
    df_m["month"] = m
    frames.append(df_m)
    print(f"  Shape: {df_m.shape}")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nTổng shape: {df_all.shape}")
print(df_all.dtypes)

Reading: C:\Users\Admin\Documents\Tuan_Huy\DREAM-2026-workshop-SCA-HPCAsia\F-DATA\data\24_01.parquet


  Shape: (705516, 14)
Reading: C:\Users\Admin\Documents\Tuan_Huy\DREAM-2026-workshop-SCA-HPCAsia\F-DATA\data\24_02.parquet
  Shape: (669758, 14)
Reading: C:\Users\Admin\Documents\Tuan_Huy\DREAM-2026-workshop-SCA-HPCAsia\F-DATA\data\24_03.parquet
  Shape: (508286, 14)
Reading: C:\Users\Admin\Documents\Tuan_Huy\DREAM-2026-workshop-SCA-HPCAsia\F-DATA\data\24_04.parquet
  Shape: (420450, 14)

Tổng shape: (2304010, 14)
embedding      object
duration      float64
avgpcon       float64
pclass         object
exit state     object
adt            object
qdt            object
cnumr           int64
nnumr           int64
pri             int64
freq_req        int64
mszl          float64
elpl          float64
month           int64
dtype: object


## 2. Train / Test split (theo tháng)

In [34]:
df_train = df_all[df_all["month"].isin([1, 2, 3])].reset_index(drop=True)
df_test  = df_all[df_all["month"] == 4].reset_index(drop=True)

print(f"Train shape : {df_train.shape}  (tháng 1–3)")
print(f"Test  shape : {df_test.shape}   (tháng 4)")

Train shape : (1883560, 14)  (tháng 1–3)
Test  shape : (420450, 14)   (tháng 4)



## 3. Chuyển đổi đơn vị target

- `duration` : giây → phút (`/ 60`)
- `avgpcon` : chia theo số node (`/ nnumr`)
- `ec` : classification — 1 nếu `"exit state"` == `"completed"`, else 0
- `pclass` : classification — 1 nếu `"compute-bound"`, else 0


In [35]:

for df in [df_train, df_test]:
    # duration: giây → phút
    df["duration"] = (df["duration"] / 60).astype(int)

    # avgpcon: chia theo số node được cấp (nnumr)
    df["avgpcon"] = (df["avgpcon"] / df["nnumr"]).astype(int)

    # ec: binary classification từ cột "exit state" gốc
    df["ec"] = (df[EXIT_STATE_COL] == "completed").astype(int)
    df.drop(columns=[EXIT_STATE_COL], inplace=True)

    # pclass: classification - compute-bound or not
    if "pclass" in df.columns:
        if df["pclass"].dtype == "object":
            df["pclass"] = (df["pclass"] == "compute-bound").astype(int)
        else:
            df["pclass"] = df["pclass"].astype(int)

print("=== Sau khi chuyển đổi đơn vị ===")
print("Train:")
print(df_train[TARGET_COLS].describe().round(2))
print("\nTest:")
print(df_test[TARGET_COLS].describe().round(2))


=== Sau khi chuyển đổi đơn vị ===
Train:
         duration     avgpcon      pclass         ec
count  1883560.00  1883560.00  1883560.00  1883560.0
mean       123.45       85.09        0.21        0.9
std        362.92       27.98        0.41        0.3
min          0.00       39.00        0.00        0.0
25%          3.00       61.00        0.00        1.0
50%         17.00       90.00        0.00        1.0
75%         70.00      104.00        0.00        1.0
max       4319.00      175.00        1.00        1.0

Test:
        duration    avgpcon     pclass         ec
count  420450.00  420450.00  420450.00  420450.00
mean      192.79      96.89       0.26       0.81
std       438.93      28.96       0.44       0.40
min         0.00      39.00       0.00       0.00
25%         3.00      85.00       0.00       1.00
50%        21.00      96.00       0.00       1.00
75%       163.00     112.00       1.00       1.00
max      4318.00     168.00       1.00       1.00


## 4. Sin/Cos encoding cho ADT và QDT

In [36]:
def cyclic_encode(df: pd.DataFrame, col: str, period: float) -> None:
    """Thêm 2 cột sin/cos cho một cột số (inplace)."""
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

for df in [df_train, df_test]:
    for col in [ADT_COL, QDT_COL]:
        dt = pd.to_datetime(df[col], errors="coerce")
        df[f"{col}_hour"]  = dt.dt.hour          # 0–23
        df[f"{col}_dow"]   = dt.dt.dayofweek     # 0=Mon … 6=Sun
        df[f"{col}_month"] = dt.dt.month         # 1–12

        cyclic_encode(df, f"{col}_hour",  24)
        cyclic_encode(df, f"{col}_dow",    7)
        cyclic_encode(df, f"{col}_month", 12)

        # Xoá cột gốc datetime và các cột int tạm
        df.drop(columns=[col,
                         f"{col}_hour", f"{col}_dow", f"{col}_month"],
                inplace=True)

# Tên các cột cyclic (dùng lại ở bước scale)
cyclic_cols = [c for c in df_train.columns
               if c.endswith("_sin") or c.endswith("_cos")]

print(f"Cyclic columns ({len(cyclic_cols)}): {cyclic_cols}")
print(df_train[cyclic_cols].describe().round(3))

Cyclic columns (12): ['adt_hour_sin', 'adt_hour_cos', 'adt_dow_sin', 'adt_dow_cos', 'adt_month_sin', 'adt_month_cos', 'qdt_hour_sin', 'qdt_hour_cos', 'qdt_dow_sin', 'qdt_dow_cos', 'qdt_month_sin', 'qdt_month_cos']
       adt_hour_sin  adt_hour_cos  adt_dow_sin  adt_dow_cos  adt_month_sin  \
count   1883560.000   1883560.000  1883560.000  1883560.000    1883560.000   
mean         -0.137        -0.113        0.009       -0.043          0.765   
std           0.675         0.716        0.685        0.728          0.212   
min          -1.000        -1.000       -0.975       -0.901          0.500   
25%          -0.866        -0.866       -0.434       -0.901          0.500   
50%          -0.259        -0.259        0.000       -0.223          0.866   
75%           0.500         0.707        0.782        0.623          1.000   
max           1.000         1.000        0.975        1.000          1.000   

       adt_month_cos  qdt_hour_sin  qdt_hour_cos  qdt_dow_sin  qdt_dow_cos  \
count

## 5. PCA trên cột embedding (95% variance)

In [37]:
# Stack embeddings thành ma trận 2-D
emb_train = np.stack(df_train[EMBEDDING_COL].values)   # (N_train, D)
emb_test  = np.stack(df_test[EMBEDDING_COL].values)    # (N_test,  D)

print(f"Embedding dim gốc: {emb_train.shape[1]}")

pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_SEED)
emb_train_pca = pca.fit_transform(emb_train)   # fit trên train
emb_test_pca  = pca.transform(emb_test)        # transform test bằng PCA đã fit

n_pca = pca.n_components_
print(f"Số chiều sau PCA (>=95% var): {n_pca}")
print(f"Cumulative variance explained: {pca.explained_variance_ratio_.sum():.4f}")

# Tạo tên cột PCA
pca_cols = [f"emb_pca_{i}" for i in range(n_pca)]

# Thêm vào dataframe, bỏ cột embedding gốc
df_train = df_train.drop(columns=[EMBEDDING_COL])
df_test  = df_test.drop(columns=[EMBEDDING_COL])

df_train[pca_cols] = emb_train_pca
df_test[pca_cols]  = emb_test_pca

print(f"\nTrain shape sau PCA: {df_train.shape}")
print(f"Test  shape sau PCA: {df_test.shape}")

Embedding dim gốc: 384
Số chiều sau PCA (>=95% var): 40
Cumulative variance explained: 0.8035

Train shape sau PCA: (1883560, 63)
Test  shape sau PCA: (420450, 63)


## 6. StandardScaler trên numeric features + PCA columns + cyclic columns

In [38]:
# cyclic_cols đã nằm trong [-1, 1] nhưng vẫn scale để đồng nhất với các bước khác
scale_cols = NUMERIC_FEATS + pca_cols + cyclic_cols

scaler = StandardScaler()
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])   # fit + transform trên train
df_test[scale_cols]  = scaler.transform(df_test[scale_cols])        # chỉ transform trên test

print(f"Đã scale {len(scale_cols)} cột")
print(f"  NUMERIC_FEATS : {NUMERIC_FEATS}")
print(f"  PCA cols      : {len(pca_cols)} cột")
print(f"  Cyclic cols   : {cyclic_cols}")

Đã scale 58 cột
  NUMERIC_FEATS : ['cnumr', 'nnumr', 'pri', 'freq_req', 'mszl', 'elpl']
  PCA cols      : 40 cột
  Cyclic cols   : ['adt_hour_sin', 'adt_hour_cos', 'adt_dow_sin', 'adt_dow_cos', 'adt_month_sin', 'adt_month_cos', 'qdt_hour_sin', 'qdt_hour_cos', 'qdt_dow_sin', 'qdt_dow_cos', 'qdt_month_sin', 'qdt_month_cos']


## 7. Lưu kết quả

In [39]:
train_path  = os.path.join(OUTPUT_DIR, "train.parquet")
test_path   = os.path.join(OUTPUT_DIR, "test.parquet")
pca_path    = os.path.join(OUTPUT_DIR, "pca.joblib")
scaler_path = os.path.join(OUTPUT_DIR, "scaler.joblib")

df_train.to_parquet(train_path, index=False)
df_test.to_parquet(test_path,   index=False)
joblib.dump(pca,    pca_path)
joblib.dump(scaler, scaler_path)

print(f"Saved train  -> {train_path}")
print(f"Saved test   -> {test_path}")
print(f"Saved PCA    -> {pca_path}")
print(f"Saved scaler -> {scaler_path}")

Saved train  -> C:\Users\Admin\Documents\Tuan_Huy\MOE\data\train.parquet
Saved test   -> C:\Users\Admin\Documents\Tuan_Huy\MOE\data\test.parquet
Saved PCA    -> C:\Users\Admin\Documents\Tuan_Huy\MOE\data\pca.joblib
Saved scaler -> C:\Users\Admin\Documents\Tuan_Huy\MOE\data\scaler.joblib


## 8. Kiểm tra nhanh

In [40]:
print("=== Train ===")
print(df_train.shape)
print(df_train.dtypes)
print(df_train.head(3))

print("\n=== Test ===")
print(df_test.shape)
print(df_test.head(3))

=== Train ===
(1883560, 63)
duration        int32
avgpcon         int32
pclass          int32
cnumr         float64
nnumr         float64
               ...   
emb_pca_35    float64
emb_pca_36    float64
emb_pca_37    float64
emb_pca_38    float64
emb_pca_39    float64
Length: 63, dtype: object
   duration  avgpcon  pclass     cnumr     nnumr       pri  freq_req  \
0         4       91       0 -0.061074 -0.061074 -0.007581  1.205362   
1         0       91       0 -0.061074 -0.061074 -0.007581 -0.829626   
2         1       91       1 -0.061074 -0.061074 -0.007581  1.205362   

       mszl      elpl  month  ...  emb_pca_30  emb_pca_31  emb_pca_32  \
0 -1.670246 -0.451894      1  ...    0.058092   -0.446513    0.623796   
1 -1.670246 -0.442739      1  ...   -1.048829   -0.042102   -1.295704   
2 -1.670246 -0.465627      1  ...    0.094025    1.253952   -1.137375   

   emb_pca_33  emb_pca_34  emb_pca_35  emb_pca_36  emb_pca_37  emb_pca_38  \
0    2.338831    1.477939   -0.255937    0.64